# VantageGro Retail Stock Insight
## Stage 4: Feature Engineering Pipeline

### Objectives
1. **Calendar Features**: Extract temporal signatures, day-of-week, weekend, quarter, and cyclical sine/cosine transformations.
2. **Historical Autoregressive Demand Lags**: Lag 1, 7, 14, 28 calculated per (Store, Product) series.
3. **Rolling Statistics & Volatility**: 7-day, 14-day, 28-day rolling means and standard deviations, strictly using `shift(1)` to eliminate target leakage.
4. **Pricing & Competitive Features**: Price ratio, price gap, and net promotional price.
5. **Generate 5 Feature Audit Visualizations** inline.
6. **Export Feature Dataset** (`vantagegro_features.csv`) and `feature_metadata.json`.

In [ ]:
import os
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

sns.set_theme(style='whitegrid', palette='viridis')
plt.rcParams['figure.figsize'] = (10, 5)
print('Feature Engineering Environment ready!')

### 1. Load Cleaned Dataset

In [ ]:
clean_path = 'data/processed/vantagegro_cleaned.csv'
if not os.path.exists(clean_path):
    clean_path = '../data/processed/vantagegro_cleaned.csv'

df = pd.read_csv(clean_path)
df['Date'] = pd.to_datetime(df['Date'])
print(f'Cleaned Data Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
df.head()

### 2. Feature Construction with Leakage Prevention

In [ ]:
data = df.sort_values(['Store ID', 'Product ID', 'Date']).reset_index(drop=True)

# Calendar Features
data['cal_year'] = data['Date'].dt.year
data['cal_month'] = data['Date'].dt.month
data['cal_day'] = data['Date'].dt.day
data['cal_day_of_week'] = data['Date'].dt.dayofweek
data['cal_is_weekend'] = data['Date'].dt.dayofweek.isin([5, 6]).astype(int)
data['cal_week_of_year'] = data['Date'].dt.isocalendar().week.astype(int)
data['cal_quarter'] = data['Date'].dt.quarter
data['sin_month'] = np.sin(2 * np.pi * data['cal_month'] / 12).round(4)
data['cos_month'] = np.cos(2 * np.pi * data['cal_month'] / 12).round(4)

# Historical Lags per Series
grouped = data.groupby(['Store ID', 'Product ID'])
data['demand_lag_1'] = grouped['Demand'].shift(1)
data['demand_lag_7'] = grouped['Demand'].shift(7)
data['demand_lag_14'] = grouped['Demand'].shift(14)
data['demand_lag_28'] = grouped['Demand'].shift(28)

# Rolling Statistics (Shifted)
data['rolling_mean_7'] = grouped['Demand'].transform(lambda s: s.shift(1).rolling(7, min_periods=1).mean()).round(2)
data['rolling_mean_14'] = grouped['Demand'].transform(lambda s: s.shift(1).rolling(14, min_periods=1).mean()).round(2)
data['rolling_mean_28'] = grouped['Demand'].transform(lambda s: s.shift(1).rolling(28, min_periods=1).mean()).round(2)
data['rolling_std_7'] = grouped['Demand'].transform(lambda s: s.shift(1).rolling(7, min_periods=1).std()).fillna(0).round(2)
data['recent_demand_trend'] = (data['rolling_mean_7'] - data['rolling_mean_28']).round(2)

# Backfill warmup
for col in ['demand_lag_1', 'demand_lag_7', 'demand_lag_14', 'demand_lag_28', 'rolling_mean_7', 'rolling_mean_14', 'rolling_mean_28', 'rolling_std_7', 'recent_demand_trend']:
    data[col] = grouped[col].bfill().ffill()

# Business Pricing Features
data['price_ratio'] = (data['Price'] / data['Competitor Pricing']).round(3)
data['discounted_price'] = (data['Price'] * (1 - data['Discount'] / 100)).round(2)
data['price_diff'] = (data['Price'] - data['Competitor Pricing']).round(2)

# Categorical Encodings
encoded_df = pd.get_dummies(data, columns=['Category', 'Region', 'Weather Condition', 'Seasonality'], drop_first=False, dtype=int)
print(f'Feature Engineering complete: {encoded_df.shape[1]} columns produced across {encoded_df.shape[0]:,} rows!')
encoded_df.head()

### 3. Feature Audit Visualizations

In [ ]:
# 1. Lag vs Target Relationship
sample = encoded_df.sample(2000, random_state=42)
plt.figure(figsize=(7, 5))
plt.scatter(sample['demand_lag_1'], sample['Demand'], alpha=0.3, color='#10b981', s=15)
plt.plot([0, 400], [0, 400], color='gray', linestyle='--')
plt.title('Lag 1 Prior-Day Demand vs Current Day Demand', fontsize=11, fontweight='bold')
plt.xlabel('Lag 1 Demand')
plt.ylabel('Target Demand')
plt.tight_layout()
plt.show()

In [ ]:
# 2. Rolling Means Trajectory (Store S001, Product P0001)
sub = encoded_df[(encoded_df['Store ID'] == 'S001') & (encoded_df['Product ID'] == 'P0001')].tail(100)
plt.figure(figsize=(11, 4.5))
plt.plot(sub['Date'], sub['Demand'], label='Actual Demand', color='#94a3b8', alpha=0.7)
plt.plot(sub['Date'], sub['rolling_mean_7'], label='7-Day Rolling Mean', color='#10b981', linewidth=2)
plt.plot(sub['Date'], sub['rolling_mean_28'], label='28-Day Rolling Mean', color='#3b82f6', linewidth=2)
plt.title('Actual Demand vs Rolling Means (Store S001, SKU P0001)', fontsize=11, fontweight='bold')
plt.legend()
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# 3. Correlation Matrix of Key Features vs Target
key_cols = ['Demand', 'demand_lag_1', 'demand_lag_7', 'rolling_mean_7', 'rolling_mean_14', 'recent_demand_trend', 'Price', 'Discount', 'Promotion', 'price_ratio']
corr = encoded_df[key_cols].corr().round(2)
plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, cmap='viridis', fmt='.2f', vmin=-1, vmax=1)
plt.title('Correlation Matrix of Key Engineered Features vs Demand', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()